# PCB Defect Detection with YOLOv8

Multiclass defect detection using **YOLOv8** (You Only Look Once) in Snowflake Notebooks with GPU acceleration.

## YOLO vs Faster R-CNN

| Aspect | YOLOv8 | Faster R-CNN |
|--------|--------|---------------|
| **Speed** | ~30-60+ FPS | ~5-7 FPS |
| **Architecture** | Single-stage | Two-stage |
| **Best For** | Real-time detection | High accuracy |
| **Inference** | Faster | More accurate on small objects |


In [ ]:
# Get active Snowflake session (container notebook)
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Verify GPU availability and container runtime
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, VRAM {props.total_memory / (1024**3):.1f} GB")


## 1. Environment Setup

Install YOLOv8 (Ultralytics) and required packages.


In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics --quiet

# Verify installation
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")


In [ ]:
# Core imports
import os
import sys
import time
import shutil
import warnings
warnings.filterwarnings("ignore")

# Data processing
import pandas as pd
import numpy as np
import base64
import io
import json
from PIL import Image

# PyTorch
import torch

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# YOLOv8
from ultralytics import YOLO

# Snowflake ML
from snowflake.ml.registry import Registry

# Set query tag for tracking
session.query_tag = {
    "origin": "sf_sit-is", 
    "name": "pcb_defect_detection_yolo",
    "version": {"major": 1, "minor": 0},
    "attributes": {"is_quickstart": 1, "source": "notebook"}
}

# Class names for PCB defects (YOLO format - 0-indexed, no background class)
CLASS_NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
NUM_CLASSES = len(CLASS_NAMES)

print(f"Number of defect classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")


## 2. Data Preparation

YOLO requires data in a specific format:
- Images in `images/train/` and `images/val/` folders
- Labels in `labels/train/` and `labels/val/` folders (`.txt` files)
- Label format: `class_id x_center y_center width height` (normalized 0-1)

We'll convert our Snowflake data to this format.


In [ ]:
# View training dataset
print("Sample training data:")
session.table("training_data").limit(5).show()


In [ ]:
# ==============================================================================
# CONVERT SNOWFLAKE DATA TO YOLO FORMAT
# ==============================================================================

def convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_width, img_height):
    """
    Convert bounding box from [xmin, ymin, xmax, ymax] to YOLO format.
    YOLO format: [x_center, y_center, width, height] (normalized 0-1)
    """
    x_center = ((xmin + xmax) / 2) / img_width
    y_center = ((ymin + ymax) / 2) / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    
    # Clamp values to [0, 1]
    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    width = max(0, min(1, width))
    height = max(0, min(1, height))
    
    return x_center, y_center, width, height


def prepare_yolo_dataset(session, output_dir='/tmp/yolo_dataset'):
    """
    Convert Snowflake training/test data to YOLO dataset format.
    
    Our dataset has CLASS values 1-6 (no background).
    YOLO expects 0-indexed classes, so we subtract 1.
    """
    # Create directory structure
    for split in ['train', 'val']:
        os.makedirs(f'{output_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{output_dir}/labels/{split}', exist_ok=True)
    
    # Process training data
    print("Loading training data from Snowflake...")
    train_df = session.table("TRAINING_DATA").to_pandas()
    print(f"  Found {len(train_df)} training records")
    
    # Process test data (as validation)
    print("Loading test data from Snowflake...")
    test_df = session.table("TEST_DATA").to_pandas()
    print(f"  Found {len(test_df)} test records")
    
    # Group by filename (multiple annotations per image)
    train_grouped = train_df.groupby('FILENAME')
    test_grouped = test_df.groupby('FILENAME')
    
    def process_split(grouped_df, split_name):
        """Process a data split (train or val)"""
        processed = 0
        skipped = 0
        
        for filename, group in grouped_df:
            # Get the first row for image data (same image for all annotations)
            first_row = group.iloc[0]
            
            try:
                # Decode and save image
                img_data = base64.b64decode(first_row['IMAGE_DATA'])
                img = Image.open(io.BytesIO(img_data)).convert('RGB')
                img_width, img_height = img.size
                
                # Save image
                img_path = f'{output_dir}/images/{split_name}/{filename}.jpg'
                img.save(img_path, 'JPEG')
                
                # Create label file with all annotations for this image
                label_path = f'{output_dir}/labels/{split_name}/{filename}.txt'
                with open(label_path, 'w') as f:
                    for _, row in group.iterrows():
                        # Convert class (1-6) to 0-indexed (0-5)
                        class_id = int(row['CLASS']) - 1
                        
                        # Skip invalid classes
                        if class_id < 0 or class_id >= NUM_CLASSES:
                            continue
                        
                        # Convert bbox to YOLO format
                        x_center, y_center, width, height = convert_bbox_to_yolo(
                            row['XMIN'], row['YMIN'], row['XMAX'], row['YMAX'],
                            img_width, img_height
                        )
                        
                        # Write YOLO format: class x_center y_center width height
                        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                
                processed += 1
                
            except Exception as e:
                skipped += 1
                if skipped <= 5:
                    print(f"  Warning: Error processing {filename}: {e}")
        
        print(f"  {split_name}: Processed {processed} images, skipped {skipped}")
        return processed
    
    # Process both splits
    train_count = process_split(train_grouped, 'train')
    val_count = process_split(test_grouped, 'val')
    
    # Create data.yaml configuration file
    yaml_content = f"""# PCB Defect Detection Dataset for YOLOv8
path: {output_dir}
train: images/train
val: images/val

# Number of classes
nc: {NUM_CLASSES}

# Class names
names: {CLASS_NAMES}
"""
    
    yaml_path = f'{output_dir}/data.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"\nDataset created at: {output_dir}")
    print(f"  Training images: {train_count}")
    print(f"  Validation images: {val_count}")
    print(f"  Config: {yaml_path}")
    
    return output_dir, yaml_path

# Prepare the dataset
DATASET_DIR, DATA_YAML = prepare_yolo_dataset(session)


## 3. Model Training with YOLOv8

YOLOv8 provides several model sizes:
- `yolov8n` - Nano (fastest, least accurate)
- `yolov8s` - Small
- `yolov8m` - Medium (good balance)
- `yolov8l` - Large
- `yolov8x` - Extra Large (slowest, most accurate)


In [ ]:
# ==============================================================================
# YOLO TRAINING CONFIGURATION
# ==============================================================================

# Training hyperparameters
EPOCHS = 50           # Number of training epochs
BATCH_SIZE = 16       # Batch size (adjust based on GPU memory)
IMG_SIZE = 640        # Image size (YOLOv8 default)
MODEL_SIZE = 'yolov8m'  # Model variant (n, s, m, l, x)

# Output directory for training results
OUTPUT_DIR = '/tmp/yolo_training'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Training Configuration:")
print(f"  Model: {MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Output: {OUTPUT_DIR}")


In [ ]:
# ==============================================================================
# TRAIN YOLOv8 MODEL
# ==============================================================================

print("Initializing YOLOv8 model...")

# Load pretrained YOLOv8 model
model = YOLO(f'{MODEL_SIZE}.pt')

print(f"Model loaded: {MODEL_SIZE}")
print(f"Starting training on {NUM_CLASSES} classes...")
print("="*70)

# Train the model
training_start = time.time()

results = model.train(
    data=DATA_YAML,           # Path to data.yaml
    epochs=EPOCHS,            # Number of epochs
    imgsz=IMG_SIZE,           # Image size
    batch=BATCH_SIZE,         # Batch size
    device=0,                 # GPU device (0 for first GPU)
    project=OUTPUT_DIR,       # Output directory
    name='pcb_defect_yolo',   # Run name
    exist_ok=True,            # Overwrite existing
    patience=10,              # Early stopping patience
    save=True,                # Save checkpoints
    plots=True,               # Generate plots
    verbose=True,             # Verbose output
    
    # Data augmentation
    augment=True,
    flipud=0.5,               # Vertical flip probability
    fliplr=0.5,               # Horizontal flip probability
    mosaic=1.0,               # Mosaic augmentation
    mixup=0.1,                # Mixup augmentation
    
    # Optimizer settings
    optimizer='AdamW',        # Optimizer
    lr0=0.001,                # Initial learning rate
    lrf=0.01,                 # Final learning rate factor
    momentum=0.937,           # Momentum
    weight_decay=0.0005,      # Weight decay
    warmup_epochs=3,          # Warmup epochs
)

training_time = time.time() - training_start

print("="*70)
print(f"Training completed!")
print(f"Total training time: {training_time:.1f}s ({training_time/60:.1f} min)")
print(f"Best model saved to: {OUTPUT_DIR}/pcb_defect_yolo/weights/best.pt")


## 4. Model Evaluation

Evaluate the trained model on the validation set and visualize results.


In [ ]:
# ==============================================================================
# LOAD BEST MODEL AND EVALUATE
# ==============================================================================

# Load the best model
best_model_path = f'{OUTPUT_DIR}/pcb_defect_yolo/weights/best.pt'
eval_model = YOLO(best_model_path)

print(f"Loaded best model from: {best_model_path}")

# Run validation
print("\nRunning validation...")
val_results = eval_model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    verbose=True,
)

print("\n" + "="*70)
print("                    VALIDATION RESULTS")
print("="*70)

# Overall metrics
print("\nOVERALL METRICS:")
print("-"*50)
print(f"  mAP@50:      {val_results.box.map50:.4f}")
print(f"  mAP@50-95:   {val_results.box.map:.4f}")
print(f"  Precision:   {val_results.box.mp:.4f}")
print(f"  Recall:      {val_results.box.mr:.4f}")

# Per-class metrics
print("\nPER-CLASS METRICS:")
print("-"*70)
print(f"{'Class':<15} {'mAP@50':<12} {'mAP@50-95':<12}")
print("-"*70)

ap50_per_class = val_results.box.ap50
ap_per_class = val_results.box.ap

for i, class_name in enumerate(CLASS_NAMES):
    if i < len(ap50_per_class):
        print(f"{class_name:<15} {ap50_per_class[i]:.4f}{'':>6} {ap_per_class[i]:.4f}")

print("-"*70)


In [ ]:
# ==============================================================================
# DISPLAY TRAINING PLOTS
# ==============================================================================
from IPython.display import Image as IPImage, display

plots_dir = f'{OUTPUT_DIR}/pcb_defect_yolo'

# Show results plot
results_plot = f'{plots_dir}/results.png'
if os.path.exists(results_plot):
    print("Training Curves:")
    display(IPImage(filename=results_plot, width=1000))

# Show confusion matrix
confusion_matrix = f'{plots_dir}/confusion_matrix.png'
if os.path.exists(confusion_matrix):
    print("\nConfusion Matrix:")
    display(IPImage(filename=confusion_matrix, width=600))

# Show F1 curve
f1_curve = f'{plots_dir}/F1_curve.png'
if os.path.exists(f1_curve):
    print("\nF1-Confidence Curve:")
    display(IPImage(filename=f1_curve, width=600))

# Show PR curve
pr_curve = f'{plots_dir}/PR_curve.png'
if os.path.exists(pr_curve):
    print("\nPrecision-Recall Curve:")
    display(IPImage(filename=pr_curve, width=600))


## 5. Test Inference

Run inference on sample images to visualize detections.


In [ ]:
# ==============================================================================
# INFERENCE UTILITIES
# ==============================================================================

def run_yolo_inference(model, image_data_b64, conf_threshold=0.25):
    """
    Run YOLO inference on a base64 encoded image.
    Returns detections in a structured format.
    """
    # Decode image
    img_data = base64.b64decode(image_data_b64)
    img = Image.open(io.BytesIO(img_data)).convert('RGB')
    
    # Run inference
    results = model(img, conf=conf_threshold, verbose=False)
    
    # Extract detections
    detections = []
    for result in results:
        boxes = result.boxes
        for i in range(len(boxes)):
            det = {
                'box': boxes.xyxy[i].cpu().numpy().tolist(),  # [x1, y1, x2, y2]
                'class_id': int(boxes.cls[i].cpu().numpy()),
                'class_name': CLASS_NAMES[int(boxes.cls[i].cpu().numpy())],
                'confidence': float(boxes.conf[i].cpu().numpy())
            }
            detections.append(det)
    
    return img, detections


def visualize_detections(img, detections, title="YOLO Detections"):
    """Visualize YOLO detections on an image."""
    # Color palette for classes
    colors = {
        'open': '#FF6B6B',
        'short': '#4ECDC4',
        'mousebite': '#45B7D1',
        'spur': '#96CEB4',
        'copper': '#FFEAA7',
        'pin-hole': '#DDA0DD'
    }
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img)
    
    for det in detections:
        x1, y1, x2, y2 = det['box']
        class_name = det['class_name']
        conf = det['confidence']
        color = colors.get(class_name, '#FF0000')
        
        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=3, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        label = f"{class_name}: {conf:.2f}"
        ax.text(x1, y1-5, label, fontsize=12, fontweight='bold',
                color='white', bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return fig

print("Inference utilities loaded!")


In [ ]:
# ==============================================================================
# TEST INFERENCE ON SAMPLE IMAGES
# ==============================================================================

print("Running inference on sample test images...")

test_df = session.table("TEST_DATA").limit(5).to_pandas()

for idx, row in test_df.iterrows():
    print(f"\nImage {idx + 1}: {row['FILENAME']}")
    print(f"  Ground Truth: Class {int(row['CLASS'])} ({CLASS_NAMES[int(row['CLASS'])-1]})")
    
    # Run inference
    img, detections = run_yolo_inference(eval_model, row['IMAGE_DATA'])
    
    print(f"  Detections: {len(detections)}")
    for det in detections:
        print(f"    - {det['class_name']}: {det['confidence']:.3f}")
    
    # Visualize
    visualize_detections(img, detections, title=f"Image: {row['FILENAME']}")


## 6. Model Deployment to Snowflake

Deploy the trained YOLO model to Snowflake Model Registry for production inference.


In [ ]:
# ==============================================================================
# CUSTOM MODEL CLASS FOR SNOWFLAKE DEPLOYMENT
# ==============================================================================

from snowflake.ml.model import custom_model

class YOLODefectDetectionModel(custom_model.CustomModel):
    """Custom model wrapper for YOLOv8 PCB defect detection."""
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self.class_names = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
    
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """
        Run YOLO inference on input images.
        
        Args:
            input_df: DataFrame with 'IMAGE_DATA' column (base64 encoded images)
            
        Returns:
            DataFrame with 'output' column containing JSON detection results
        """
        import base64
        import io
        import json
        from PIL import Image
        from ultralytics import YOLO
        
        # Load model from context
        model = self.context.model_ref('yolo')
        
        results_list = []
        
        for _, row in input_df.iterrows():
            # Decode image
            img_data = base64.b64decode(row['IMAGE_DATA'])
            img = Image.open(io.BytesIO(img_data)).convert('RGB')
            
            # Run inference
            results = model(img, conf=0.25, verbose=False)
            
            # Extract detections
            detections = {
                'boxes': [],
                'labels': [],
                'scores': [],
                'class_names': []
            }
            
            for result in results:
                boxes = result.boxes
                for i in range(len(boxes)):
                    detections['boxes'].append(boxes.xyxy[i].cpu().numpy().tolist())
                    detections['labels'].append(int(boxes.cls[i].cpu().numpy()))
                    detections['scores'].append(float(boxes.conf[i].cpu().numpy()))
                    detections['class_names'].append(self.class_names[int(boxes.cls[i].cpu().numpy())])
            
            results_list.append(json.dumps(detections))
        
        return pd.DataFrame({'output': results_list})

print("Custom model class defined!")


In [ ]:
# ==============================================================================
# LOG MODEL TO SNOWFLAKE REGISTRY
# ==============================================================================

# Load the trained YOLO model
best_model_path = f'{OUTPUT_DIR}/pcb_defect_yolo/weights/best.pt'
yolo_model = YOLO(best_model_path)

# Create the custom model instance
yolo_custom_model = YOLODefectDetectionModel(
    context=custom_model.ModelContext(
        models={'yolo': yolo_model}
    )
)

# Prepare sample input
sample_df = session.table("TEST_DATA").limit(1).to_pandas()
sample_input = pd.DataFrame({'IMAGE_DATA': [sample_df.iloc[0]['IMAGE_DATA']]})
sample_spdf = session.create_dataframe(sample_input)

# Initialize registry
ml_reg = Registry(session=session)

# Model and service names
MODEL_NAME = "YOLODefectDetectionModel"
SERVICE_NAME = "YOLODEFECTDETECTSERVICE"

print("Cleaning up existing resources...")
session.sql(f"DROP SERVICE IF EXISTS PCB_CV.PUBLIC.{SERVICE_NAME}").collect()

# Drop any existing MODEL_BUILD services for YOLO
try:
    services_df = session.sql("SHOW SERVICES IN SCHEMA PCB_CV.PUBLIC").collect()
    for svc in services_df:
        svc_name = svc["name"]
        if svc_name.startswith("MODEL_BUILD_") and "YOLO" in svc_name.upper():
            session.sql(f"DROP SERVICE IF EXISTS PCB_CV.PUBLIC.{svc_name}").collect()
            print(f"  Dropped service: {svc_name}")
except:
    pass

# Delete existing model
try:
    ml_reg.delete_model(MODEL_NAME)
    print(f"  Deleted existing model: {MODEL_NAME}")
except Exception as e:
    print(f"  No existing model to delete")

# Log the model
print(f"\nLogging model to registry: {MODEL_NAME}...")

mv = ml_reg.log_model(
    yolo_custom_model,
    model_name=MODEL_NAME,
    version_name='v1',
    pip_requirements=["ultralytics", "torch", "torchvision"],
    sample_input_data=sample_spdf,
    options={
        "embed_local_ml_library": True,
        "relax": True,
        "use_gpu": True,
        "cuda_version": "12.6"
    },
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"]
)

print(f"Model logged successfully: {MODEL_NAME} v1")


In [ ]:
# ==============================================================================
# DEPLOY MODEL AS SPCS SERVICE
# ==============================================================================

print(f"Creating inference service: {SERVICE_NAME}...")

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool="PCB_CV_SERVICE_COMPUTEPOOL",
    ingress_enabled=False,
    image_repo="PCB_CV.PUBLIC.IMAGE_REPO",
    max_instances=1,
    gpu_requests=1
)

print(f"Service {SERVICE_NAME} created successfully!")
print("\nNote: Service may take a few minutes to become ready.")


## 7. Run Remote Inference

Test the deployed YOLO model service.


In [ ]:
# ==============================================================================
# TEST REMOTE INFERENCE
# ==============================================================================

print("Testing remote YOLO inference...")

# Get model reference
m = ml_reg.get_model(MODEL_NAME)
mv = m.version("v1")

# Prepare test input
test_df = session.table("TEST_DATA").limit(1).to_pandas()
image_data_df = pd.DataFrame({'IMAGE_DATA': [test_df.iloc[0]['IMAGE_DATA']]})

# Run remote prediction
print("Running inference via SPCS service...")
remote_prediction = mv.run(
    image_data_df,
    function_name="predict",
    service_name=SERVICE_NAME
)

print("\nRemote Prediction Result:")
print(remote_prediction)


In [ ]:
# ==============================================================================
# VISUALIZE REMOTE INFERENCE RESULTS
# ==============================================================================

# Parse and visualize the results
output_str = remote_prediction.iloc[0]['output']
detections_dict = json.loads(output_str)

print(f"Detected {len(detections_dict['boxes'])} objects:")
for i, (box, label, score, name) in enumerate(zip(
    detections_dict['boxes'],
    detections_dict['labels'],
    detections_dict['scores'],
    detections_dict['class_names']
)):
    print(f"  {i+1}. {name} (class {label}): {score:.3f}")

# Visualize
img_data = base64.b64decode(test_df.iloc[0]['IMAGE_DATA'])
img = Image.open(io.BytesIO(img_data)).convert('RGB')

# Convert to detection format for visualization
vis_detections = []
for box, label, score, name in zip(
    detections_dict['boxes'],
    detections_dict['labels'],
    detections_dict['scores'],
    detections_dict['class_names']
):
    vis_detections.append({
        'box': box,
        'class_id': label,
        'class_name': name,
        'confidence': score
    })

visualize_detections(img, vis_detections, title="YOLO Remote Inference Result")


## 8. Summary

### What We Built

| Component | Details |
|-----------|---------|
| **Model** | YOLOv8-Medium |
| **Architecture** | Single-stage object detection |
| **Classes** | 6 PCB defect types |
| **Speed** | ~30-60+ FPS (much faster than R-CNN) |
| **Deployment** | Snowflake ML Registry + SPCS |

### Key Differences from Faster R-CNN

| Aspect | YOLO | Faster R-CNN |
|--------|------|--------------|
| Detection | Single pass | Two-stage |
| Speed | ~10x faster | More thorough |
| Accuracy | Slightly lower | Higher on small objects |
| Use Case | Real-time inspection | Accuracy-critical tasks |


In [ ]:
# ==============================================================================
# FINAL SUMMARY
# ==============================================================================

print("\n" + "="*60)
print("     YOLOv8 PCB DEFECT DETECTION - COMPLETE!")
print("="*60)
print(f"\n  Model: YOLOv8-{MODEL_SIZE[6:].upper()}")
print(f"  Classes: {NUM_CLASSES} defect types")
print(f"  Training Time: {training_time/60:.1f} minutes")
print(f"  mAP@50: {val_results.box.map50:.4f}")
print(f"  mAP@50-95: {val_results.box.map:.4f}")
print(f"\n  Best Model: {best_model_path}")
print(f"  Registry: {MODEL_NAME} v1")
print(f"  Service: {SERVICE_NAME}")
print("\n" + "="*60)
print("\nNext Steps:")
print("  1. Use the Streamlit app for interactive inference")
print("  2. Compare results with Faster R-CNN notebook")
print("  3. Fine-tune hyperparameters for better accuracy")
print("="*60)
